# 3. Macro and micro drivers: why growth, not only the budget, decides the debt

**Questions.** Why has Italy grown less than its partners since the 1990s, and what does that imply for the debt? How much of the gap in
GDP per capita comes from productivity, hours, employment and demographics? What role do interest rates, inflation, the labour market,
innovation and the structure of firms play? What would closing some of these gaps be worth for the public finances?

**Data.** Eurostat national accounts and labour market (`nama_10_gdp`, `nama_10_pe`, `demo_pjanind`, `lfsi_emp_a`), R&D
(`rd_e_gerdtot`), government bond yields (`irt_lt_mcby_a`), government finance (`gov_10dd_edpt1`).

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "public_debt").is_dir())
try:
    import public_debt
except ImportError:  # not installed: use the source folder (the extension must be built in place)
    sys.path.insert(0, str(ROOT / "scripts" / "public_debt"))
    import public_debt
from public_debt import accounting, data, dsa, fiscal, growth, synthetic
from public_debt.var import fit_var1

core = public_debt.require_cpp()

DATA_MODE = os.environ.get("PUBLIC_DEBT_DATA_MODE", "official")   # "official" (Eurostat, IMF) or "synthetic" (offline)
OFFICIAL = DATA_MODE == "official"
OUTPUT_DIR = ROOT / "outputs" / "growth_drivers"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"public_debt {public_debt.__version__} | data mode: {DATA_MODE}")
if not OFFICIAL:
    print("WARNING: synthetic data. Tables and charts are placeholders, not statistics about Italy.")

## 3.1 Growth: Italy versus its partners

In [ ]:
PEERS = ("IT", "DE", "FR", "ES", "EU27_2020")
if OFFICIAL:
    fiscal_by = {g: data.fiscal_dataset(g) for g in PEERS}
    labour_by = {g: data.labour_and_population(g) for g in PEERS if not g.startswith("EU")}
    ind = data.structural_indicators(PEERS)
    yields = data.bond_yields(PEERS)
else:
    fiscal_by = {g: synthetic.fiscal_dataset(seed=20 + k) for k, g in enumerate(PEERS)}
    labour_by = {g: synthetic.labour_and_population(seed=30 + k) for k, g in enumerate(PEERS) if not g.startswith("EU")}
    ind, yields = synthetic.structural_indicators(PEERS), synthetic.bond_yields()
real = pd.DataFrame({g: (1 + f["real_growth"].fillna(0)).cumprod() for g, f in fiscal_by.items()})
real = 100 * real / real.iloc[0]
fig, ax = plt.subplots(figsize=(10, 4))
real.plot(ax=ax)
ax.set(title=f"Real GDP, index {real.index[0]} = 100", ylabel="index")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "real_gdp.png", dpi=150)
avg = pd.DataFrame({g: 100 * f["real_growth"].groupby(pd.cut(f.index, [1995, 2007, 2013, 2019, 2100],
                                                              labels=["1996-2007", "2008-2013", "2014-2019", "2020-latest"])).mean()
                    for g, f in fiscal_by.items()})
print("Average real GDP growth by period (%):")
display(avg.round(2))

## 3.2 Growth accounting

$\Delta\ln(Y/P) = \Delta\ln(Y/H) + \Delta\ln(H/E) + \Delta\ln(E/W) + \Delta\ln(W/P)$: productivity per hour, hours per worker,
employment rate of the working-age population and the working-age share. Average annual contributions in percentage points.

In [ ]:
periods = {"1996-2007": (1996, 2007), "2008-2019": (2008, 2019), "2020-latest": (2020, 2100)}
tables = {g: growth.average_by_period(growth.decompose(lp), periods) for g, lp in labour_by.items()}
acc = pd.concat(tables, names=["country", "period"])
display(acc.round(2))
fig, ax = plt.subplots(figsize=(11, 4))
acc.drop(columns="GDP per capita").xs("1996-2007", level="period").plot.bar(ax=ax, stacked=True, alpha=0.8, position=1, width=0.35)
acc.drop(columns="GDP per capita").xs("2008-2019", level="period").plot.bar(ax=ax, stacked=True, alpha=0.45, position=0, width=0.35,
                                                                             legend=False)
ax.axhline(0, color="k", lw=0.8)
ax.set(title="Contributions to GDP per capita growth (pp per year): left 1996-2007, right 2008-2019", xlabel="")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "growth_accounting.png", dpi=150)

## 3.3 Labour market, demographics and innovation

In [ ]:
latest = {name: panel.dropna(how="all").iloc[-1] for name, panel in ind.items()}
snap = pd.DataFrame(latest).T
snap.index = ["employment rate 20-64 (%)", "female employment rate 20-64 (%)", "old-age dependency ratio (%)", "R&D, % of GDP"]
display(snap[[g for g in PEERS if g in snap.columns]].round(1))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, key, title in zip(axes, ["employment_rate", "old_age_dependency", "rd_intensity"],
                          ["Employment rate 20-64 (%)", "Old-age dependency ratio (%)", "R&D expenditure (% of GDP)"]):
    ind[key].plot(ax=ax, legend=key == "employment_rate")
    ax.set(title=title, xlabel="")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "structural_indicators.png", dpi=150)

## 3.4 Interest rates, spreads and inflation

The spread between Italian and German 10-year yields is the market price of Italy's fiscal and growth risk; it feeds the effective rate on
the debt as bonds are refinanced. Inflation lowers the debt ratio (through nominal GDP) but raises rates.

In [ ]:
spread = (yields["IT"] - yields["DE"]).dropna()
it = fiscal_by["IT"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
yields[[c for c in ("IT", "DE", "FR", "ES") if c in yields]].plot(ax=axes[0])
axes[0].set(title="10-year government bond yields (%)", xlabel="")
axes[1].bar(spread.index, 100 * spread, color="C3", alpha=0.7)
axes[1].set(title="Italy - Germany spread (basis points, annual average)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "spreads.png", dpi=150)
windows = {"1999-2007": (1999, 2007), "2008-2013": (2008, 2013), "2014-2019": (2014, 2019), "2020-latest": (2020, 2100)}
print("Average spread by period (bp): " + ", ".join(f"{k}: {100 * spread.loc[a:b].mean():.0f}"
                                                    for k, (a, b) in windows.items() if not spread.loc[a:b].empty))

## 3.5 What would closing the gaps be worth for the debt?

Illustrative level effects, translated into public finances with the current revenue-to-GDP ratio and fed into the C++ debt projection:

- **Employment**: raising the employment rate to the EU average over ten years, with new workers 30% less productive than the average.
- **Productivity**: 0.5 percentage points more productivity growth per year.

Extra GDP raises revenues roughly in proportion (tax-to-GDP ratio); part of the extra revenue may be spent, so we assume that 50% improves the
primary balance. The debt effect combines the larger denominator and the better primary balance.

In [ ]:
it = fiscal_by["IT"]
d0, i0 = float(it["debt"].iloc[-1]), float(it["effective_rate"].iloc[-1])
revenue_ratio = 0.47
emp_gap = float(snap.loc["employment rate 20-64 (%)", "EU27_2020"] - snap.loc["employment rate 20-64 (%)", "IT"]) / 100
horizon = 10
level_emp = max(emp_gap, 0) / max(float(snap.loc["employment rate 20-64 (%)", "IT"]) / 100, 1e-6) * 0.7
extra_growth_emp = (1 + level_emp) ** (1 / horizon) - 1
base = dsa.Scenario.constant(int(it.index.max()) + 1, horizon, growth=0.008, inflation=0.02,
                             market_rate=float(yields["IT"].dropna().iloc[-1]) / 100,
                             primary_balance=float(it["primary_balance"].iloc[-1]))

def with_growth(extra):
    level = np.cumprod(np.full(horizon, 1 + extra)) - 1
    return base.shifted(growth=extra, primary_balance=0.5 * revenue_ratio * level / (1 + level))

runs = {"baseline": base, "employment at EU average": with_growth(extra_growth_emp),
        "productivity +0.5 pp a year": with_growth(0.005)}
proj = {k: dsa.project(d0, i0, 1 / 7, s)["debt"] for k, s in runs.items()}
table = pd.DataFrame({k: 100 * v for k, v in proj.items()})
display(table.iloc[[0, 4, 9]].round(1).T.rename(columns=lambda y: f"debt {y} (%)"))
print(f"Employment gap with the EU: {100 * emp_gap:.1f} points; implied extra growth {100 * extra_growth_emp:.2f} pp a year for {horizon} years")

## 3.6 Micro foundations of low growth

Aggregate productivity reflects how firms, markets and institutions work. The evidence for Italy points to:

- **Firm size and management.** A very large share of micro and small firms, often family-managed, with lower adoption of technology and
  management practices; resources are slow to move to the most productive firms (Bank of Italy studies; Bloom and Van Reenen on
  management practices; Eurostat structural business statistics, `sbs_sc_ovw`).
- **Institutions.** Long civil-justice proceedings and administrative burdens reduce investment and firm growth (CEPEJ reports; OECD
  Economic Surveys of Italy).
- **Human capital and innovation.** Low tertiary attainment and R&D intensity compared with peers (section 3.3), skill mismatches and the
  emigration of graduates.
- **Labour market participation.** Low employment of women and young people and large territorial gaps between North and South.
- **Tax compliance.** A large tax gap (the Ministry of Economy's annual report on the non-observed economy and tax evasion estimates it at
  several points of GDP) narrows the tax base and raises statutory rates for compliant taxpayers, with a high tax wedge on labour.
- **Demographics.** Population ageing raises pension and health spending and lowers potential growth (section 3.3; Ageing Report of the
  European Commission).

These factors are the channels through which "structural reforms" can lower the debt ratio: they act on the denominator and on the tax
base, and they are the focus of the National Recovery and Resilience Plan (PNRR).

### References

- Bloom, N. and Van Reenen, J. (2010). Why do management practices differ across firms and countries? *Journal of Economic Perspectives*, 24(1), 203-224.
- European Commission, *Ageing Report* (triennial); OECD, *Economic Surveys: Italy*.
- Ministero dell'Economia e delle Finanze, *Relazione sull'economia non osservata e sull'evasione fiscale e contributiva* (annual).
- Banca d'Italia, *Relazione annuale*; Eurostat, national accounts and labour force statistics.